In [1]:
import os
import certifi
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [2]:
from langchain.agents import create_react_agent, AgentExecutor

In [3]:
# Load environment variables from .env file
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [4]:
search_tool = TavilySearchResults(max_results=2)

In [5]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}: {data}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [6]:
# LLM
llm = ChatOpenAI(
    model = "gpt-3.5-turbo",
    temperature = 0,
    api_key = OPENAI_API_KEY
)

In [7]:
response = llm.invoke("Tell me a joke about AI")
response

AIMessage(content='Why did the AI break up with its robot girlfriend? Because she had too many bugs in her system!', response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 13, 'total_tokens': 34, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-fdb0904d-7426-4bbe-8923-68eb41954264-0')

In [8]:
# Prompt
prompt = hub.pull("hwchase17/react")

c:\Users\sudee\OneDrive\Desktop\End to End Single Agent Project\singleagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [9]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [10]:
# Tool
tools = [search_tool, get_weather_data]

In [11]:
# Create the agent
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [12]:
# Executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

In [13]:
# Run
response = agent_executor.invoke({
    "input": (
        "Find the capital of sri lanka"
        "and then find its current weather."
    )
})




> Entering new AgentExecutor chain...
I should first search for the capital of Sri Lanka and then use the weather data tool to find its current weather.
Action: tavily_search_results_json
Action Input: "capital of Sri Lanka"[{'url': 'https://en.wikipedia.org/wiki/Capital_of_Sri_Lanka', 'content': 'Tools\n\nActions\n\n Read\n Edit\n View history\n\nGeneral\n\n What links here\n Related changes\n Upload file\n Permanent link\n Page information\n Cite this page\n Get shortened URL\n Switch to legacy parser\n\nPrint/export\n\n Download as PDF\n Printable version\n\nIn other projects\n\n Wikidata item\n\nAppearance\n\nFrom Wikipedia, the free encyclopedia\n\nThe current legislative capital of Sri Lanka is Sri Jayawardenepura Kotte and the executive and judicial capital is Colombo. Over the course of the island\'s history, the national capital has been in several locations.\n\n| Part of a series on the | [...] #### Kandyan period (1597–1815)\n\n[edit")]\n\n Kandy, 1597–1815\n\n#### British

In [14]:
print(response["output"])

The capital of Sri Lanka is Sri Jayawardenepura Kotte and the current weather there is 25°C with light rain showers.
